# JavaScript — Array methods

## LESSON 29 — forEach, map, filter, find, reduce

LESSON 25-26 covered what an array holds and how to reorder it. These five methods are about **walking** it. They replace most `for` loops, and each one takes a **function** that runs once per item.

| method | gives back | use it when |
|---|---|---|
| `forEach` | **nothing** (`undefined`) | you just want to do something with each item |
| `map` | a **new array**, same length | you want to transform every item |
| `filter` | a **new array**, shorter or equal | you want to keep only some items |
| `find` | **one item**, or `undefined` | you want the first item that matches |
| `reduce` | **one single value** | you want to collapse the array into a total |

```js
const numbers = [1, 2, 3, 4];

numbers.forEach((n) => console.log(n));      // prints, returns nothing
numbers.map((n) => n * 2);                   // [2, 4, 6, 8]
numbers.filter((n) => n > 2);                // [3, 4]
numbers.find((n) => n > 2);                  // 3     (just the first)
numbers.reduce((total, n) => total + n, 0);  // 10
```

The middle column is the one to learn. Nearly every mistake in this lesson is expecting one of those shapes and getting another.

### Who calls your function

You write the function, the **method** calls it — once per item, handing the item in. That is the callback idea from LESSON 21, and it means the function does not have to be written on the spot:

```js
const double = (n) => n * 2;

numbers.map(double);             // [2, 4, 6, 8]
numbers.map((n) => double(n));   // [2, 4, 6, 8]   -> the same answer, spelled out
```

`map(double)` hands `map` the function itself. `map((n) => double(n))` hands it a new function that calls `double`. Both work here — LESSON 30 shows a case where they do not.

### forEach cannot be stopped

`forEach` always runs to the end. `break` is a **syntax error** inside it, because `break` belongs to a loop and a callback is a function:

```js
numbers.forEach((n) => {
  if (n > 2) break;   // SyntaxError: Illegal break statement
});
```

`return` is allowed, but it does less than it looks. It ends **this one call**, so the method simply moves on to the next item:

```js
numbers.forEach((n) => {
  if (n > 2) return;   // skips this item only, does not stop anything
  console.log(n);      // 1, then 2
});
```

To really stop early, use `for...of` from LESSON 17 — `break` works there — or ask the question with `find` or `some` instead.

### map always returns something

`map` builds the new array out of whatever your function gives back. Give back nothing and you get an array of `undefined`:

```js
numbers.map((n) => n * 2);       // [2, 4, 6, 8]
numbers.map((n) => { n * 2 });   // [undefined, undefined, undefined, undefined]
```

The braces turn the arrow body into a block, and a block needs the word `return`. Drop the braces, or write `return n * 2;` — LESSON 21 has the two arrow shapes side by side.

### filter gives an array, find gives the item

Same question, two shapes of answer — and they part company when nothing matches.

```js
numbers.filter((n) => n > 100);   // []          -> an empty array
numbers.find((n) => n > 100);     // undefined
```

**An empty array is truthy.** `[]` is still an array, so this test passes even on a miss:

```js
if (numbers.filter((n) => n > 100)) {   // always true
  console.log("found something");       // prints, with nothing found
}
```

Ask about the length instead: `if (numbers.filter((n) => n > 100).length > 0)`.

`find` has the opposite trap. It gives you the item **or** `undefined`, and reading a property of `undefined` throws — the same "reading *through* a missing value" rule as LESSON 27:

```js
const products = [{ name: "Mouse", price: 25 }];

const found = products.find((p) => p.price > 1000);

found;        // undefined
found.name;   // TypeError: Cannot read properties of undefined (reading 'name')
```

Check before you read: `if (found) { console.log(found.name); }`. LESSON 32 has a one-character tool for this, `?.`.

### reduce, slowly

This is the trap of the lesson, and the method people avoid for months. It is worth ten minutes now.

`reduce` takes two things: a function `(accumulator, item)` and a **starting value**.

```js
numbers.reduce((total, n) => total + n, 0);
//               ^acc   ^item          ^start at 0
```

Each round, whatever the function **returns** becomes the next `total`. Round by round over `[1, 2, 3, 4]`:

| round | `total` coming in | `n` | what it returns |
|---|---|---|---|
| 1 | `0` — the starting value | `1` | `1` |
| 2 | `1` | `2` | `3` |
| 3 | `3` | `3` | `6` |
| 4 | `6` | `4` | `10` |

The value returned by the last round is the result. Two rules hide in that table.

**1. The function must return the next accumulator.** Forget the `return` and round 2 receives `undefined`:

```js
numbers.reduce((total, n) => { total + n; }, 0);   // undefined
```

The braces again — the same mistake as `map`, with a quieter result.

**2. The starting value is not really optional.** Leave it out and the **first item** becomes the starting accumulator, so round 1 is skipped:

```js
numbers.reduce((total, n) => total + n);   // 10   -> right answer, by luck
[].reduce((total, n) => total + n);        // TypeError: Reduce of empty array with no initial value
[].reduce((total, n) => total + n, 0);     // 0    -> the starting value answers for the empty case
```

An array you did not write yourself can always turn out empty. Pass the starting value.

### reduce is not only for sums

The accumulator can be any shape you want. A number, to keep the largest item:

```js
numbers.reduce((best, n) => (n > best ? n : best), 0);   // 4
//                           ^ the ternary from LESSON 13
```

Or an **object**, counting how often each value appears — the object keys of LESSON 27, filled one round at a time:

```js
const votes = ["yes", "no", "yes"];

votes.reduce((counts, vote) => {
  if (counts[vote] === undefined) counts[vote] = 0;
  counts[vote] = counts[vote] + 1;
  return counts;                        // rule 1: hand the accumulator on
}, {});                                 // { yes: 2, no: 1 }
```

`counts[vote]` needs brackets because the key lives in a variable — case 1 from LESSON 27.

### With arrays of objects

This is the shape you'll meet constantly in real apps — and in React.

```js
const products = [
  { name: "Laptop", price: 1200 },
  { name: "Mouse", price: 25 },
];

products.filter((p) => p.price < 100);              // [{ name: "Mouse", price: 25 }]
products.map((p) => p.name);                        // ["Laptop", "Mouse"]
products.reduce((total, p) => total + p.price, 0);  // 1225
```

### Key notes

- **`forEach` returns nothing.** `const result = arr.forEach(...)` gives `undefined`. If you want an array back, you wanted `map`.
- **`forEach` cannot be stopped.** `break` is a syntax error inside it, and `return` only skips one item. Use `for...of` (LESSON 17), `find` or `some` when you need to stop early.
- `map` and `filter` **never modify** the original array — they build a new one. You must store or use the result.
- The function must **return** the value. `(n) => n * 2` returns implicitly; `(n) => { n * 2 }` returns `undefined` — in `map` and in `reduce` alike.
- `filter` gives you an **array** (possibly with one item). `find` gives you the **item itself**. `[item]` and `item` are not the same thing.
- **`filter` gives `[]` on a miss, and `[]` is truthy.** Test `.length`, never the array itself. **`find` gives `undefined` on a miss**, so check it before reading a property off it.
- **Always pass `reduce` a starting value.** Without one the first item takes its place, and an empty array throws `TypeError: Reduce of empty array with no initial value`.

### Example

In [ ]:
const exampleNumbers = [1, 2, 3, 4];

exampleNumbers.forEach((n) => console.log("item:", n));

console.log(exampleNumbers.map((n) => n * 2));
console.log(exampleNumbers.filter((n) => n > 2));
console.log(exampleNumbers.find((n) => n > 2));
console.log(exampleNumbers.reduce((total, n) => total + n, 0));

// The original array is untouched
console.log(exampleNumbers);

// A named function works just as well as one written on the spot
const exampleDouble = (n) => n * 2;
console.log(exampleNumbers.map(exampleDouble));

// forEach cannot be stopped: `return` skips one item, it does not break out
exampleNumbers.forEach((n) => {
  if (n > 2) return;
  console.log("not skipped:", n);
});

// `break` in there is a SyntaxError, and a SyntaxError stops the WHOLE cell
// from running — uncomment the next line and nothing above it prints either.
// exampleNumbers.forEach((n) => { if (n > 2) break; });

// The braces swallow the return
console.log(exampleNumbers.map((n) => { n * 2 }));

// Nothing matches: filter gives an empty array, find gives undefined
console.log(exampleNumbers.filter((n) => n > 100));
console.log(exampleNumbers.find((n) => n > 100));

// ...and [] is truthy, which is why you ask about the length instead
console.log(Boolean(exampleNumbers.filter((n) => n > 100)));
console.log(exampleNumbers.filter((n) => n > 100).length > 0);

// reduce with no starting value works here, by luck...
console.log(exampleNumbers.reduce((total, n) => total + n));
// ...and not at all on an empty array — uncomment to see the TypeError
// console.log([].reduce((total, n) => total + n));
console.log([].reduce((total, n) => total + n, 0));

// Forgetting the return inside reduce
console.log(exampleNumbers.reduce((total, n) => { total + n; }, 0));

// An accumulator that is not a running total: the largest item...
console.log(exampleNumbers.reduce((best, n) => (n > best ? n : best), 0));

// ...and an object counting how often each value appears
const exampleVotes = ["yes", "no", "yes"];

console.log(
  exampleVotes.reduce((counts, vote) => {
    if (counts[vote] === undefined) counts[vote] = 0;
    counts[vote] = counts[vote] + 1;
    return counts;
  }, {}),
);

const exampleProducts = [
  { name: "Laptop", price: 1200 },
  { name: "Mouse", price: 25 },
  { name: "Keyboard", price: 80 },
];

console.log(exampleProducts.map((p) => p.name));
console.log(exampleProducts.filter((p) => p.price < 100));
console.log(exampleProducts.find((p) => p.name === "Mouse"));
console.log(exampleProducts.reduce((total, p) => total + p.price, 0));

// find on a miss gives undefined, so check before reading a property off it
const exampleMissing = exampleProducts.find((p) => p.price > 5000);

console.log(exampleMissing);
// console.log(exampleMissing.name);   // uncomment to see the TypeError

if (exampleMissing) {
  console.log(exampleMissing.name);
} else {
  console.log("nothing over 5000");
}

### Exercise

Given:

```js
const prices = [10, 25, 4, 60, 33];
```

1. Print a new array with every price **doubled**.
2. Print a new array with only the prices **above 20**.
3. Print the **first** price above 20.
4. Print the **sum** of all prices.
5. Print the **highest** price, using `reduce`.
6. Print what `prices.forEach((price) => price * 2)` hands back, and say why in a comment.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here


### Mini challenge

Given:

```js
const users = [
  { name: "Sam", age: 34, active: true },
  { name: "Alex", age: 17, active: false },
  { name: "Mia", age: 28, active: true },
];
```

1. Print an array with just the **names**.
2. Print only the users who are **active**.
3. Print the user named `"Mia"` (the object itself, not an array).
4. Print the **total** of all the ages.
5. Build `{ Sam: 34, Alex: 17, Mia: 28 }` from the users with `reduce`, and print it.

In [ ]:
// Your code here


## LESSON 30 — More iteration methods

LESSON 29 covered the five you will use every day. These are the rest, plus the two things that make all of them more useful.

| method | gives back | use it when |
|---|---|---|
| `some` | `true` / `false` | is **at least one** item like this? |
| `every` | `true` / `false` | are **all** items like this? |
| `findIndex` | index, or `-1` | where is the first match? |
| `flat` | a flatter array | you have arrays inside an array |
| `flatMap` | mapped, then flattened | each item produces several results |

```js
const ages = [17, 22, 30];

ages.some((age) => age >= 18);      // true   -> at least one adult
ages.every((age) => age >= 18);     // false  -> not all of them
ages.findIndex((age) => age >= 18); // 1
```

`some` and `every` stop as soon as they know the answer.

### The extra callback parameters

Every one of these methods hands the callback three things: the item, its **index**, and the whole array.

```js
["a", "b"].map((item, index) => `${index}: ${item}`);   // ["0: a", "1: b"]
```

Take only what you need. Most of the time that is just the item.

### Flattening

```js
[[1, 2], [3, 4]].flat();                    // [1, 2, 3, 4]
[[1, [2]], [3]].flat(2);                    // [1, 2, 3]      -> depth 2

const orders = [{ items: ["pen", "book"] }, { items: ["lamp"] }];
orders.flatMap((order) => order.items);     // ["pen", "book", "lamp"]
```

### Chaining

Each of `map`, `filter` and `flat` returns an array, so the next call can follow straight on.

```js
products
  .filter((p) => p.inStock)
  .map((p) => p.name)
  .join(", ");
```

Read it top to bottom: keep the ones in stock, take their names, join them. Each line is one step, and one thing to check when the result is wrong.

### Key notes

- **`some` and `every` return a boolean, not items.** If you wanted the matching items, you wanted `filter`.
- **`every` on an empty array is `true`.** There is no item that fails the test. This is correct and still surprises people.
- `findIndex` returns `-1` when nothing matches — and `-1` is truthy, so test it with `!== -1`.
- Chaining creates an array at every step. That is fine for the sizes you will meet, and clarity is worth more than the saving.

### Example

In [ ]:
const exampleAges = [17, 22, 30];

console.log(exampleAges.some((age) => age >= 18));
console.log(exampleAges.every((age) => age >= 18));
console.log(exampleAges.findIndex((age) => age >= 18));
console.log([].every((age) => age >= 18));

console.log(["a", "b"].map((item, index) => `${index}: ${item}`));

console.log([[1, 2], [3, 4]].flat());
console.log([[1, [2]], [3]].flat(2));

const exampleOrders = [{ items: ["pen", "book"] }, { items: ["lamp"] }];
console.log(exampleOrders.flatMap((order) => order.items));

const exampleProducts = [
  { name: "Laptop", inStock: true },
  { name: "Mouse", inStock: false },
  { name: "Lamp", inStock: true },
];

console.log(
  exampleProducts
    .filter((product) => product.inStock)
    .map((product) => product.name)
    .join(", "),
);

### Exercise

Given:

```js
const orders = [
  { id: 1, total: 40, paid: true },
  { id: 2, total: 15, paid: false },
  { id: 3, total: 90, paid: true },
];
```

1. Print `true` if **any** order is unpaid.
2. Print `true` if **all** orders are above 10.
3. Print the index of the first order above 50.
4. Print `"1: 40, 2: 15, 3: 90"` in one chain, using the id and the total.
5. Print the total value of the paid orders only.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Given:

```js
const classes = [
  { name: "A", students: ["mia", "alex"] },
  { name: "B", students: ["sam"] },
  { name: "C", students: [] },
];
```

1. Print every student name as one flat array.
2. Print the names of the classes that have at least one student.
3. Print `true` if every class has fewer than five students.
4. Print each class as `"A (2)"`, joined by `" | "`.

In [ ]:
// Your code here